Overview
Medical Image Classification with MONAI: MedNIST Tutorial

Level: Beginner

This notebook teaches how to classify medical images (X-rays, CT scans, MRIs) into categories using deep learning. We'll use MONAI (a PyTorch-based medical imaging library) and the MedNIST dataset.

In [ ]:
!pip install -q "monai[pillow]" matplotlib scikit-learn

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from monai.apps import MedNISTDataset
from torch.utils.data import Dataset
from monai.networks.nets import DenseNet121
from monai.transforms import (
    Compose,
    LoadImage,
    EnsureChannelFirst,
    ScaleIntensity,
    Resize,
    RandRotate90,
    RandFlip,
)
from monai.utils import set_determinism

set_determinism(seed=0)
random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu" )
print(f"Using device: {device}")



1. Load and Explore the Data

MedNIST contains 64x64 grayscale images across 6 categories: AbdomenCT, BreastMRI, CXR, ChestCT, Hand, HeadCT. Always look at your data before writing any model code.

In [ ]:
root_dir = "./mednist_data"
os.makedirs(root_dir, exist_ok=True)

raw_train = MedNISTDataset(root_dir=root_dir, section="training", download=True, seed=0)

class_names = sorted(list({item["class_name"] for item in raw_train.data}))
print("Classes found:", class_names)
print("Number of training examples:", len(raw_train))

In [ ]:
image_files = [item ["image"] for item in raw_train.data]
labels = [item["class_name"] for item in raw_train.data]

fig, axes = plt.subplots(2,4, figsize=(12, 6))
axes = axes.flatten()
sample_idxs = random.sample(range(len(image_files)), 8)

for ax, idx in zip(axes, sample_idxs):
  from PIL import Image
  img = Image.open(image_files[idx])
  ax.imshow(img, cmap="gray")
  ax.set_title(labels[idx])
  ax.axis("off")

plt.tight_layout()
plt.show()

2. Preprocessing

Raw images need to be resized, normalized, and converted to tensors before a model can use them. We also add random rotation/flipping (only during training) so the model generalizes better instead of memorizing images.

In [ ]:
train_transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),
    ScaleIntensity(),
    Resize((64, 64)),
    RandRotate90(prob=0.5, spatial_axes=(0, 1)),
    RandFlip(prob=0.5, spatial_axis=0),
])

eval_transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),
    ScaleIntensity(),
    Resize((64, 64)),
])

In [ ]:
test_img = train_transforms(image_files[0])
print("Shape after transform:", test_img.shape)

3. Train / Validation / Test Split

We use MedNIST's pre-made splits: training data (what the model learns from), validation (checked during training), and test (touched only once, at the end).

In [ ]:
class MedNISTClassificationDataset(Dataset):
    """Wraps MedNIST file paths + labels into a MONAI Dataset with our transforms applied."""

    def __init__(self, data_list, class_names, transforms):
        self.image_files = [d["image"] for d in data_list]
        self.labels = [class_names.index(d["class_name"]) for d in data_list]
        self.transforms = transforms

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        image = self.transforms(self.image_files[index])
        label = self.labels[index]
        return image, label


raw_val = MedNISTDataset(root_dir=root_dir, section="validation", download=False, seed=0)
raw_test = MedNISTDataset(root_dir=root_dir, section="test", download=False, seed=0)

train_ds = MedNISTClassificationDataset(raw_train.data, class_names, train_transforms)
val_ds = MedNISTClassificationDataset(raw_val.data, class_names, eval_transforms)
test_ds = MedNISTClassificationDataset(raw_test.data, class_names, eval_transforms)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_ds)} | Validation: {len(val_ds)} | Test: {len(test_ds)}")

4. Build the Model

We use MONAI's DenseNet121, a well-tested CNN architecture, adapted for grayscale 2D medical images.

In [ ]:
model = DenseNet121(
    spatial_dims=2,
    in_channels=1,
    out_channels=len(class_names),
).to(device)

loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print(model)

5. Train the Model

Standard training loop: predict → compute loss → backpropagate → update weights, repeated over 5 epochs.

In [ ]:
NUM_EPOCHS = 5

train_losses = []
val_accuracies = []

for epoch in range(NUM_EPOCHS):
    # --- Training ---
    model.train()
    epoch_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- Validation ---
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train loss: {avg_train_loss:.4f} | Val accuracy: {val_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, NUM_EPOCHS + 1), train_losses, marker="o")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(range(1, NUM_EPOCHS + 1), val_accuracies, marker="o", color="green")
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")

plt.tight_layout()
plt.show()

6. Evaluate on the Test Set

The real, honest measure of performance — data the model never saw during training.

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

test_acc = (np.array(all_preds) == np.array(all_labels)).mean()
print(f"Test accuracy: {test_acc:.4f}\n")
from sklearn.metrics import classification_report
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()
sample_idxs = random.sample(range(len(test_ds)), 8)

for ax, idx in zip(axes, sample_idxs):
    image, true_label = test_ds[idx]
    pred_label = all_preds[idx]

    ax.imshow(image[0].numpy(), cmap="gray")
    correct = pred_label == true_label
    color = "green" if correct else "red"
    ax.set_title(
        f"True: {class_names[true_label]}\nPred: {class_names[pred_label]}",
        color=color, fontsize=9
    )
    ax.axis("off")

plt.tight_layout()
plt.show()